In [1]:
import random
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
import ruptures as rpt


# ------------------------
# 設定
# ------------------------
NUM_COINS = 200
DATA_DIR = Path.cwd() / "coingecko_by_coin"
OUTPUT_SUMMARY_CSV = Path.cwd() / "change_point_summary.csv"
OUTPUT_PAIRS_CSV = Path.cwd() / "change_point_pairs.csv"

AR_LAGS = 5
CHANGE_MODEL = "rbf"

# 変化点を今より多めに検出しやすくする設定
CHANGE_PEN_BASE = 1.0
MIN_CHANGE_DISTANCE = 2
CHANGE_MIN_SIZE = 2
CHANGE_JUMP = 1

PRICE_COL_CANDIDATES = ["price", "Price", "close", "Close"]
TIME_COL_CANDIDATES = ["timestamp", "Timestamp", "date", "Date", "time", "Time"]


def log(msg: str):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")


# ------------------------
# CSV一覧を取得
# ------------------------
def find_coin_files(data_dir: Path):
    if not data_dir.exists():
        raise FileNotFoundError(f"フォルダが見つかりません: {data_dir}")

    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"CSVファイルが見つかりません: {data_dir}")

    return files


# ------------------------
# 列名を自動判定
# ------------------------
def detect_column(df: pd.DataFrame, candidates: list, col_type: str):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(
        f"{col_type}列が見つかりません。候補: {candidates}, 実際の列: {list(df.columns)}"
    )


# ------------------------
# CSV読み込み
# ------------------------
def load_coin_csv(file_path: Path):
    df = pd.read_csv(file_path)

    time_col = detect_column(df, TIME_COL_CANDIDATES, "時刻")
    price_col = detect_column(df, PRICE_COL_CANDIDATES, "価格")

    df = df[[time_col, price_col]].copy()
    df.columns = ["timestamp", "price"]

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    df = df.dropna(subset=["timestamp", "price"]).copy()
    df = df[df["price"] > 0].copy()
    df = df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    if len(df) == 0:
        raise ValueError(f"有効なデータがありません: {file_path.name}")

    return df


# ------------------------
# ファイル名から銘柄名を推定
# ------------------------
def symbol_from_filename(file_path: Path):
    return file_path.stem


# ------------------------
# 対数収益率を作成
# ------------------------
def make_log_returns(df: pd.DataFrame):
    prices = df["price"].values.astype(float)

    if len(prices) < 2:
        return pd.DataFrame(columns=["timestamp", "log_return"])

    log_returns = np.diff(np.log(prices))
    return_times = df["timestamp"].iloc[1:].reset_index(drop=True)

    ret_df = pd.DataFrame({
        "timestamp": return_times,
        "log_return": log_returns
    })

    ret_df = ret_df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return ret_df


# ------------------------
# AR を対数収益率に適用
# ------------------------
def apply_ar_returns_in_sample(ret_df: pd.DataFrame, symbol: str, lags: int = 5):
    series = pd.Series(ret_df["log_return"].values.astype(float))

    if len(series) <= lags + 5:
        log(f"[{symbol}] 対数収益率データが短すぎるため AR をスキップします")
        return None, None

    model = AutoReg(series, lags=lags, old_names=False)
    res = model.fit()

    fitted = res.fittedvalues
    actual = series.iloc[lags:]

    min_len = min(len(actual), len(fitted))
    actual = actual.iloc[:min_len]
    fitted = fitted.iloc[:min_len]

    mse = float(((actual.values - fitted.values) ** 2).mean())
    rmse = float(np.sqrt(mse))

    log(f"[{symbol}] AR({lags}) on returns: MSE={mse:.6e}, RMSE={rmse:.6e}")
    return mse, rmse


# ------------------------
# 変化点の間引き
# ------------------------
def filter_change_points_by_distance(bkps, min_distance):
    filtered = []
    for b in bkps:
        if not filtered or (b - filtered[-1] >= min_distance):
            filtered.append(b)
    return filtered


# ------------------------
# 変化点検知
# ------------------------
def detect_change_points(
    ret_df: pd.DataFrame,
    symbol: str,
    model_name="rbf",
    pen_base=1.0,
    min_distance=2,
    min_size=2,
    jump=1
):
    returns = ret_df["log_return"].values.astype(float)

    if len(returns) < 10:
        log(f"[{symbol}] 変化点検知用データが短すぎるためスキップします")
        return [], [], np.array([]), pd.Series(dtype="datetime64[ns]")

    std = returns.std()
    if std < 1e-12:
        log(f"[{symbol}] 収益率の分散がほぼ0のため変化点検知をスキップします")
        return [], [], returns, ret_df["timestamp"]

    returns_scaled = (returns - returns.mean()) / (std + 1e-8)
    signal = returns_scaled.reshape(-1, 1)

    pen_value = pen_base * np.log(len(signal))

    algo = rpt.Pelt(
        model=model_name,
        min_size=min_size,
        jump=jump
    ).fit(signal)

    bkps = algo.predict(pen=pen_value)

    valid_bkps = [b for b in bkps if b < len(returns)]
    valid_bkps = filter_change_points_by_distance(valid_bkps, min_distance)

    log(f"[{symbol}] pen_value={pen_value:.6f}")
    log(f"[{symbol}] change points (raw): {bkps}")
    log(f"[{symbol}] change points (filtered): {valid_bkps}")

    return bkps, valid_bkps, returns_scaled, ret_df["timestamp"]


# ------------------------
# 変化点候補の表示
# ------------------------
def summarize_change_points(ret_df: pd.DataFrame, symbol: str, valid_bkps):
    print(f"\n=== {symbol} の変化点候補 ===")

    if not valid_bkps:
        print("変化点は検出されませんでした。")
        return

    for i, b in enumerate(valid_bkps, start=1):
        if b < len(ret_df):
            t = ret_df["timestamp"].iloc[b]
            r = ret_df["log_return"].iloc[b]
            print(f"{i}. {t.strftime('%Y-%m-%d %H:%M:%S')} 付近, log_return={r:.6f}")


# ------------------------
# 変化点の日付抽出
# ------------------------
def extract_change_point_dates(ret_df: pd.DataFrame, valid_bkps):
    dates = []
    for b in valid_bkps:
        if b < len(ret_df):
            dates.append(ret_df["timestamp"].iloc[b])
    return dates


# ------------------------
# 変化点強度抽出
# ------------------------
def extract_change_point_strengths(ret_df: pd.DataFrame, valid_bkps):
    returns = ret_df["log_return"].values.astype(float)
    strengths = []

    for b in valid_bkps:
        if 1 <= b < len(returns):
            strength = abs(returns[b] - returns[b - 1])
        elif b == 0 and len(returns) > 0:
            strength = abs(returns[b])
        else:
            strength = np.nan
        strengths.append(float(strength) if pd.notna(strength) else np.nan)

    return strengths


# ------------------------
# {銘柄, 日付} ペアを作成
# ------------------------
def build_symbol_date_pairs(change_point_summary: dict):
    pairs = []

    for symbol, rows in change_point_summary.items():
        for row in rows:
            pairs.append({
                "symbol": symbol,
                "date": row["date"],
                "strength": row["strength"],
            })

    pairs.sort(key=lambda x: (x["date"], x["symbol"]))
    return pairs


# ------------------------
# {銘柄, 日付} 形式で表示
# ------------------------
def print_pair_format(change_point_summary: dict):
    print("\n==============================")
    print("変化点一覧 {銘柄, 日付}")
    print("==============================")

    pairs = build_symbol_date_pairs(change_point_summary)

    if not pairs:
        print("データなし")
        return

    for row in pairs:
        print(f"{{{row['symbol']}, {row['date'].strftime('%Y-%m-%d %H:%M:%S')}}}")


# ------------------------
# 銘柄別サマリーDataFrame作成
# ------------------------
def build_summary_dataframe(results: list):
    rows = []

    for r in results:
        cp_dates = r.get("change_points", [])
        cp_strengths = r.get("change_strengths", [])

        cp_dates_str = " | ".join(
            [d.strftime("%Y-%m-%d %H:%M:%S") for d in cp_dates]
        ) if cp_dates else ""

        cp_strengths_str = " | ".join(
            [f"{x:.6f}" for x in cp_strengths]
        ) if cp_strengths else ""

        rows.append({
            "symbol": r.get("symbol"),
            "rows": r.get("rows"),
            "return_rows": r.get("return_rows"),
            "start_time": r.get("start_time"),
            "end_time": r.get("end_time"),
            "min_price": r.get("min_price"),
            "max_price": r.get("max_price"),
            "last_price": r.get("last_price"),
            "mse": r.get("mse"),
            "rmse": r.get("rmse"),
            "n_change_points": len(cp_dates),
            "change_points": cp_dates_str,
            "change_strengths": cp_strengths_str,
        })

    return pd.DataFrame(rows)


# ------------------------
# {銘柄, 日付} 用DataFrame作成
# ------------------------
def build_pairs_dataframe(change_point_summary: dict):
    pairs = build_symbol_date_pairs(change_point_summary)

    if not pairs:
        return pd.DataFrame(columns=["symbol", "date", "strength"])

    df_pairs = pd.DataFrame(pairs)
    df_pairs["date"] = pd.to_datetime(df_pairs["date"], errors="coerce")
    return df_pairs


# ------------------------
# ランキング表示
# ------------------------
def print_rankings(summary_df: pd.DataFrame):
    print("\n==============================")
    print("ランキング")
    print("==============================")

    valid_rmse = summary_df.dropna(subset=["rmse"]).sort_values("rmse")
    if len(valid_rmse) > 0:
        print("\n[AR RMSE が小さい上位10件]")
        for i, (_, row) in enumerate(valid_rmse.head(10).iterrows(), start=1):
            print(f"{i}. {row['symbol']}  RMSE={row['rmse']:.6f}")

        print("\n[AR RMSE が大きい上位10件]")
        for i, (_, row) in enumerate(
            valid_rmse.sort_values("rmse", ascending=False).head(10).iterrows(),
            start=1
        ):
            print(f"{i}. {row['symbol']}  RMSE={row['rmse']:.6f}")
    else:
        print("\nRMSEを計算できた銘柄がありません。")

    cp_rank = summary_df.sort_values("n_change_points", ascending=False)
    if len(cp_rank) > 0:
        print("\n[変化点が多い上位10件]")
        for i, (_, row) in enumerate(cp_rank.head(10).iterrows(), start=1):
            print(f"{i}. {row['symbol']}  変化点数={int(row['n_change_points'])}")


# ------------------------
# メイン処理
# ------------------------
def main():
    random.seed(42)
    np.random.seed(42)

    coin_files = find_coin_files(DATA_DIR)

    if NUM_COINS > len(coin_files):
        raise ValueError(f"NUM_COINS={NUM_COINS} は CSV数 {len(coin_files)} を超えています。")

    log(f"=== {DATA_DIR} からランダムに {NUM_COINS} 個選択 ===")

    selected_files = random.sample(coin_files, NUM_COINS)

    print(f"選ばれた{NUM_COINS}ファイル:")
    for f in selected_files:
        print(" -", f.name)

    dfs = {}
    returns_map = {}
    results = []
    change_point_summary = {}

    # ------------------------
    # データ読み込み
    # ------------------------
    log("=== データ読み込み開始 ===")
    for idx, file_path in enumerate(selected_files, start=1):
        symbol = symbol_from_filename(file_path)

        try:
            log(f"({idx}/{NUM_COINS}) Loading {file_path.name} ...")
            df = load_coin_csv(file_path)
            ret_df = make_log_returns(df)

            dfs[symbol] = df
            returns_map[symbol] = ret_df

            log(
                f"[{symbol}] rows={len(df)} "
                f"period={df['timestamp'].iloc[0]} -> {df['timestamp'].iloc[-1]}"
            )
        except Exception as e:
            log(f"[{symbol}] 読み込み失敗: {e}")

    if not dfs:
        log("読み込めた銘柄がありません。終了します。")
        return None, None

    # ------------------------
    # AR + 変化点検知
    # ------------------------
    log("=== AR・変化点検知開始 ===")
    for symbol, df in dfs.items():
        ret_df = returns_map[symbol]

        mse = None
        rmse = None
        cp_dates = []
        cp_strengths = []

        try:
            mse, rmse = apply_ar_returns_in_sample(ret_df, symbol, lags=AR_LAGS)
        except Exception as e:
            log(f"[{symbol}] AR失敗: {e}")

        try:
            bkps, valid_bkps, returns_scaled, return_times = detect_change_points(
                ret_df,
                symbol,
                model_name=CHANGE_MODEL,
                pen_base=CHANGE_PEN_BASE,
                min_distance=MIN_CHANGE_DISTANCE,
                min_size=CHANGE_MIN_SIZE,
                jump=CHANGE_JUMP,
            )
            summarize_change_points(ret_df, symbol, valid_bkps)
            cp_dates = extract_change_point_dates(ret_df, valid_bkps)
            cp_strengths = extract_change_point_strengths(ret_df, valid_bkps)

            change_point_summary[symbol] = [
                {"date": d, "strength": s}
                for d, s in zip(cp_dates, cp_strengths)
            ]
        except Exception as e:
            log(f"[{symbol}] 変化点検知失敗: {e}")
            change_point_summary[symbol] = []

        results.append({
            "symbol": symbol,
            "rows": len(df),
            "return_rows": len(ret_df),
            "start_time": df["timestamp"].iloc[0].strftime("%Y-%m-%d %H:%M:%S"),
            "end_time": df["timestamp"].iloc[-1].strftime("%Y-%m-%d %H:%M:%S"),
            "min_price": float(df["price"].min()),
            "max_price": float(df["price"].max()),
            "last_price": float(df["price"].iloc[-1]),
            "mse": mse,
            "rmse": rmse,
            "change_points": cp_dates,
            "change_strengths": cp_strengths,
        })

    # ------------------------
    # DataFrame化
    # ------------------------
    summary_df = build_summary_dataframe(results)
    pairs_df = build_pairs_dataframe(change_point_summary)

    # ------------------------
    # 銘柄別サマリー表示
    # ------------------------
    print("\n==============================")
    print("銘柄別サマリー")
    print("==============================")
    print(summary_df)

    # ------------------------
    # ランキング
    # ------------------------
    print_rankings(summary_df)

    # ------------------------
    # 最後に {銘柄, 日付} 形式で出力
    # ------------------------
    print_pair_format(change_point_summary)

    # ------------------------
    # DataFrame表示
    # ------------------------
    print("\n==============================")
    print("summary_df")
    print("==============================")
    print(summary_df)

    print("\n==============================")
    print("pairs_df")
    print("==============================")
    print(pairs_df)

    # ------------------------
    # CSV保存
    # ------------------------
    try:
        summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")
        log(f"summary_df を保存しました: {OUTPUT_SUMMARY_CSV}")
    except Exception as e:
        log(f"summary_df 保存失敗: {e}")

    try:
        if len(pairs_df) > 0:
            pairs_df_to_save = pairs_df.copy()
            pairs_df_to_save["date"] = pairs_df_to_save["date"].dt.strftime("%Y-%m-%d %H:%M:%S")
            pairs_df_to_save.to_csv(OUTPUT_PAIRS_CSV, index=False, encoding="utf-8-sig")
            log(f"pairs_df を保存しました: {OUTPUT_PAIRS_CSV}")
        else:
            log("pairs_df は空のため CSV 保存をスキップしました。")
    except Exception as e:
        log(f"pairs_df 保存失敗: {e}")

    return summary_df, pairs_df


if __name__ == "__main__":
    summary_df, pairs_df = main()

[2026-04-14 22:40:52] === d:\musashino-university\finance\coingecko_by_coin からランダムに 200 個選択 ===
選ばれた200ファイル:
 - cocoro_cocoro.csv
 - akta_akita-inu-asa.csv
 - meai_meai.csv
 - kenji_kenji.csv
 - hopecoin_hopecoin-2.csv
 - diemlibre_dlb.csv
 - chess_tranchess.csv
 - cad_caduceus-protocol.csv
 - sos_strategic-oil-supply-2.csv
 - anima_apeiron-anima.csv
 - ampmnta_eris-amplified-mnta.csv
 - catalorian_catalorian.csv
 - hela_hela.csv
 - ilum_illuminati.csv
 - wvic_wrapped-viction.csv
 - alpay_alpha-pay.csv
 - gene_genopets.csv
 - solpaka_solpaka.csv
 - hiss_snake-of-solana.csv
 - tmoon_thermo-fisher-scientific-ondo-tokenized-stock.csv
 - mendi_mendi-finance.csv
 - $node_nodelyai.csv
 - elys_elysionyx-by-virtuals.csv
 - sov_sovryn.csv
 - peipei_peipeicoin-vip.csv
 - eden_edenlayer.csv
 - hai_hyzen-ai.csv
 - paw_paw-v2.csv
 - cat_cat-inu.csv
 - retsba_retsba.csv
 - celo-australian-dollar_audm.csv
 - ppw_pentagon-pizza-watch.csv
 - perk_perk.csv
 - loop_workloop-ai.csv
 - asdai_aave-v3-sdai.c